# Lab 09 — Market Basket — Groceries
**Unsupervised Track** · Intermediate · ~60 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Encode transactions for association rules
2. Compute support, confidence, and lift
3. Generate Apriori-style candidate itemsets
4. Filter rules by lift > 2

## Datasets (this folder)
- `groceries.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-09-market-basket-groceries/lab-09-market-basket-groceries.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Paste the lab token from the ViLabs page into `VL_TOKEN` in **Cell 0**, then run that cell first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell. It checks the logic of your function against small cases. A different implementation is fine. It does not require the same lines of code or an exact number. If a check fails, the lab page says what is wrong and what to change. It does not give the answer.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`groceries.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-09-market-basket-groceries"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-09-market-basket-groceries/bundle/dataset.zip"
NEED = ["groceries.csv"]  # unzipped files used by the code below


import json as _json
import urllib.parse as _urlparse
import urllib.request as _urlrequest

VLABS_API = "https://prod-api.vilabs.app"
VL_TOKEN = ""  # paste the lab token shown on the ViLabs page after Start Lab

def vlabs_submit():
    """Validation function. Do not modify."""
    import inspect
    token = str(VL_TOKEN or "").strip()
    if not token:
        print("ViLabs: paste the lab token into VL_TOKEN in Cell 0, then re-run this cell.")
        return None
    names = ["freq_pairs", "rv_confidence", "tf_lift"]
    answers = {}
    missing = []
    for name in names:
        fn = globals().get(name)
        if not callable(fn):
            missing.append(name)
            continue
        answers[name] = inspect.getsource(fn)
    if missing:
        print("ViLabs: write these functions before validating:", ", ".join(missing))
        return None
    base = VLABS_API.rstrip("/") + "/colab/labs/" + LAB_ID + "/session"
    ua = {"User-Agent": "Mozilla/5.0 (compatible; ViLabs-Colab/1.0)"}

    def _read(req):
        try:
            with _urlrequest.urlopen(req, timeout=90) as resp:
                return _json.loads(resp.read().decode())
        except _urlrequest.HTTPError as exc:
            print("ViLabs HTTP", exc.code, exc.read().decode(errors="replace")[:300])
            return None

    payload = _json.dumps({"token": token, "answers": answers}).encode()
    result = _read(_urlrequest.Request(
        base + "/validate/",
        data=payload,
        headers={"Content-Type": "application/json", **ua},
    ))
    if result is None:
        return None
    print("ViLabs validated", result.get("validated"))
    for row in result.get("variables") or []:
        print(row.get("name"), "pass" if row.get("ok") else "fail")
    return result

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Retail Analytics: Market Basket Analysis on Grocery Receipts

> **Scenario:** A supermarket chain has scanned 9,835 checkout baskets and wants to know which products are bought together. The answers drive real business decisions: which items to place on adjacent shelves, which products to bundle in promotions, and what to recommend at the checkout. You will implement association-rule mining (the Apriori idea) in pure Python — no mlxtend — to discover these patterns.
>
> **You will learn:** parsing headerless variable-length transaction data, transaction encoding, the three rule metrics (support, confidence, lift), Apriori-style candidate generation, and how to filter rules for actionable insights.
> **Time:** ~60 minutes. **Level:** Intermediate. **Needs:** pandas + matplotlib. **Env:** 🟢 Colab only.

### Mental Map

| Concept | Where it appears | Why it matters |
|---|---|---|
| Transaction encoding | Section 1 | Baskets → sets of items; the format all rule mining needs |
| Support | Section 2 | How often does this itemset appear? (frequency) |
| Confidence | Section 2 | If A is bought, how often is B also bought? (reliability) |
| Lift | Section 2 | Is the co-occurrence stronger than chance? (interestingness) |
| Apriori principle | Section 2 | If {A,B} is rare, {A,B,C} must be rarer — prune candidates |
| Rule filtering | Section 3 | Lift > 2 finds genuinely surprising associations |

---

## 1. Parsing Transactions and Transaction Encoding

The `groceries.csv` file is **headerless** and **variable-length**: each row is one basket, items are comma-separated, and baskets range from 1 to 32 items. pandas `read_csv` fails on this format because it expects a fixed number of columns. We parse it with the `csv` module instead, converting each row to a `frozenset` of items — a set because order within a basket doesn't matter, and frozen because it can be hashed and counted.


In [ ]:
import csv
from collections import Counter
from itertools import combinations

# Parse the headerless, variable-length CSV
transactions = []
with open("groceries.csv", newline="", encoding="utf-8") as f:
    for row in csv.reader(f):
        items = frozenset(item.strip() for item in row if item.strip())
        transactions.append(items)

N = len(transactions)
print(f"Number of baskets: {N}")
basket_sizes = [len(t) for t in transactions]
print(f"Basket size — min: {min(basket_sizes)}, max: {max(basket_sizes)}, "
      f"mean: {sum(basket_sizes)/N:.2f}")

# Count individual item frequencies
item_counts = Counter(item for t in transactions for item in t)
print(f"Unique items: {len(item_counts)}")
print("\nTop 10 most frequent items:")
for item, count in item_counts.most_common(10):
    print(f"  {item:25s} {count:5d}  support={count/N:.4f}")


**What to notice:**
1. There are 9,835 baskets with 169 unique items; the average basket holds 4.41 items.
2. `whole milk` appears in 2,513 baskets (support ≈ 0.2555) — it's in one of every four baskets.
3. Using `frozenset` lets us test subset relationships (`itemset <= transaction`) which is the core operation of support counting.

> **Pitfall:** Not stripping whitespace from item names. The file contains entries like `"cream cheese "` with a trailing space — without `.strip()`, `"cream cheese"` and `"cream cheese "` would count as different items.

---

## 2. Support, Confidence, Lift, and the Apriori Principle

An **association rule** has the form `A → B` ("customers who buy A also buy B"). Three metrics judge a rule:

- **Support** = fraction of all baskets containing both A and B. High support = common pattern.
- **Confidence** = P(B | A) = support(A∪B) / support(A). High confidence = reliable rule.
- **Lift** = confidence / support(B) = P(A∩B) / (P(A)·P(B)). Lift > 1 means A and B are positively correlated; lift > 2 is genuinely interesting.

The **Apriori principle** makes search tractable: if an itemset is infrequent, all its supersets are also infrequent. So we build candidates level by level — first frequent 1-itemsets, then 2-itemsets whose 1-itemset subsets are all frequent, and so on — pruning the search space dramatically.


In [ ]:
def support(itemset):
    """Fraction of transactions containing all items in itemset."""
    return sum(1 for t in transactions if itemset <= t) / N

MIN_SUPPORT = 0.01

freq_1 = {frozenset([item]) for item, c in item_counts.items() if c / N >= MIN_SUPPORT}
print(f"Frequent 1-itemsets (support >= {MIN_SUPPORT}): {len(freq_1)}")

candidate_2 = Counter()
for t in transactions:
    for a, b in combinations(sorted(t), 2):
        if frozenset([a]) in freq_1 and frozenset([b]) in freq_1:
            candidate_2[frozenset([a, b])] += 1

freq_2 = {s for s, c in candidate_2.items() if c / N >= MIN_SUPPORT}
print(f"Frequent 2-itemsets: {len(freq_2)}")

def rules_from_itemset(itemset):
    """Yield (antecedent, consequent, support, confidence, lift) for a 2-itemset."""
    for a in itemset:
        b = next(iter(itemset - frozenset([a])))
        sup = candidate_2[itemset] / N
        conf = candidate_2[itemset] / item_counts[a]
        lift = conf / (item_counts[b] / N)
        yield (a, b, sup, conf, lift)

all_rules = []
for s in freq_2:
    all_rules.extend(rules_from_itemset(s))

print(f"\nTotal 2-itemset rules generated: {len(all_rules)}")
print("\nTop 10 rules by confidence:")
for a, b, sup, conf, lift in sorted(all_rules, key=lambda r: -r[3])[:10]:
    print(f"  {a:20s} -> {b:20s}  sup={sup:.4f}  conf={conf:.4f}  lift={lift:.4f}")


**What to notice:**
1. 88 frequent 1-itemsets and 213 frequent 2-itemsets survive the 1% support threshold — the Apriori prune eliminates millions of candidates.
2. Each frequent 2-itemset generates 2 rules (A→B and B→A), giving 426 total rules before filtering.
3. Confidence alone is misleading: a rule pointing to a very popular item (like `whole milk`) can have high confidence but lift ≈ 1, meaning it's not actually interesting.

> **Pitfall:** Confusing confidence with lift. `butter → whole milk` has confidence 0.50 but lift only 1.95 — it's common because `whole milk` is everywhere, not because butter buyers are special. Lift corrects for the consequent's baseline popularity.

---

## 3. Filtering Rules for Actionable Insights

Raw rule lists are overwhelming. The art of market basket analysis is **filtering** for the rules that matter. Three practical filters:

1. **Consequent filter:** "What leads to whole milk?" — useful for shelf placement.
2. **Lift filter:** Keep only rules with lift > 2 — genuinely surprising associations.
3. **Support floor:** Require minimum support so rules are based on enough data to trust.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

wm_rules = [(a, b, sup, conf, lift) for a, b, sup, conf, lift in all_rules
            if b == "whole milk" and conf >= 0.10]
wm_rules.sort(key=lambda r: -r[3])
print(f"Rules -> whole milk (conf >= 0.10): {len(wm_rules)}")
print("Top 10 by confidence:")
for a, b, sup, conf, lift in wm_rules[:10]:
    print(f"  {a:20s} -> whole milk  sup={sup:.4f}  conf={conf:.4f}  lift={lift:.4f}")

high_lift = [(a, b, sup, conf, lift) for a, b, sup, conf, lift in all_rules
             if lift > 2.0]
high_lift.sort(key=lambda r: -r[4])
print(f"\nRules with lift > 2: {len(high_lift)}")
print("Top 15 by lift:")
for a, b, sup, conf, lift in high_lift[:15]:
    print(f"  {a:20s} -> {b:20s}  sup={sup:.4f}  conf={conf:.4f}  lift={lift:.4f}")

# --- Visualize: support vs confidence, point size = lift ---
sups = [r[2] for r in all_rules]
confs = [r[3] for r in all_rules]
lifts = [r[4] for r in all_rules]
plt.figure(figsize=(7, 5))
plt.scatter(sups, confs, s=[l * 8 for l in lifts], alpha=0.5, c=lifts, cmap="viridis")
plt.colorbar(label="Lift")
plt.xlabel("Support")
plt.ylabel("Confidence")
plt.title("Association Rules (size/color = lift)")
plt.savefig("rules_scatter.png", dpi=100)
from IPython.display import display
display(plt.gcf())
print("Saved rules_scatter.png")


**What to notice:**
1. 45 rules have `whole milk` as the consequent with confidence ≥ 0.10; the strongest is `butter → whole milk` (conf ≈ 0.50, lift ≈ 1.95).
2. 44 rules have lift > 2; the strongest is `root vegetables → beef` (lift ≈ 3.04) — root vegetable buyers are 3× more likely to buy beef than a random shopper.
3. The scatter plot shows most rules cluster at low support and moderate confidence; the high-lift rules (bright yellow) are the actionable gems.

> **Pitfall:** Acting on low-support, high-lift rules. A rule with support 0.001 (10 baskets) might have lift 5 but be pure noise. Always pair lift with a minimum support threshold.

---

## Exercises

Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell. It checks the logic of your function against small cases. A different implementation is fine. It does not require the same lines of code or an exact number. If a check fails, the lab page says what is wrong and what to change. It does not give the answer.

### Exercise 1 — Top 10 Itemsets by Support
Compute the support of every 1-itemset and every 2-itemset. Report the top 10 1-itemsets and the top 10 2-itemsets by support.


**Follow-up:** An itemset is frequent when its support is at least 1 percent. Return how many 2-itemsets meet that threshold. Do not count single items.

### Exercise 2 — Rules for "Whole Milk" Consequent
Generate all rules where the consequent is `whole milk` (min support 0.01, min confidence 0.10). Report the count and the top rule by confidence.


**Follow-up:** Among baskets that contain root vegetables, what fraction also contain whole milk? Return the confidence of root vegetables to whole milk, not the lift and not the reverse rule.

### Exercise 3 — High-Lift Rules Only
Filter all 2-itemset rules to those with lift > 2. Report the count and the top rule by lift.


**Follow-up:** Compare how often tropical fruit and pip fruit appear together with how often they would if they were independent. Return the lift of tropical fruit to pip fruit.

<details>
<summary>Hint</summary>

For Exercise 3, iterate over `freq_2`, and for each itemset `{a, b}` generate both directions: `a → b` and `b → a`. Compute lift as `conf / (item_counts[b] / N)` and keep rules where lift > 2.

</details>

---


In [ ]:
def freq_pairs():
    """An itemset is frequent when its support is at least 1 percent. Return how many 2-itemsets meet that threshold. Do not count single items."""
    pass


In [ ]:
def rv_confidence():
    """Among baskets that contain root vegetables, what fraction also contain whole milk? Return the confidence of root vegetables to whole milk, not the lift and not the reverse rule."""
    pass


In [ ]:
def tf_lift():
    """Compare how often tropical fruit and pip fruit appear together with how often they would if they were independent. Return the lift of tropical fruit to pip fruit."""
    pass


In [ ]:
vlabs_submit()


### What to learn next
- Extend Apriori to 3-itemsets and watch the candidate count explode — then try the FP-Growth algorithm, which avoids candidate generation entirely.
- Implement the FP-Tree data structure and compare its runtime to Apriori on this dataset.
- Explore the `mlxtend` library's `apriori` and `association_rules` functions and verify they match your pure-Python results.
- Apply association rules to a recommendation engine: "Customers who bought X also bought Y."
- Move to Course 3 (Deep Learning) and learn how neural collaborative filtering discovers these patterns automatically from raw purchase history.

*Files in this folder: groceries.csv.*


---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
